# Taxi IA Chapinero — Regresión lineal múltiple

**Tercer corte.** Predicción del tiempo derivado por arista OSM y franja semanal, sobre el dataset operativo auditado. El objetivo es evaluar un modelo de aproximación a los escenarios generados; no confundir estas métricas con validación de tráfico real.

## 1. Dataset y variables
- Unidad de análisis: una arista OSM dirigida × día de semana × hora.
- Objetivo: `tiempo_min` derivado de longitud y velocidad.
- Predictores: `distancia_m`, codificación cíclica de `hora`, `dia_semana` y `tipo_via`.
- Excluimos `velocidad_kmh` porque interviene directamente en el cálculo del objetivo.
- Partición **por aristas** disjuntas para evitar memorizar exactamente la misma arista entre train y test.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
RAIZ = Path.cwd()
if not (RAIZ / "backend").exists():
    RAIZ = RAIZ.parents[1] if RAIZ.name == "notebooks" else RAIZ
entrada=RAIZ/"backend/datos/ml/red_osm_auditada/dataset_red_osm_operativo_auditado.csv.gz"
df=pd.read_csv(entrada)
print("Filas, columnas:",df.shape)
print("Aristas:",df.indice_arista_osm.nunique())
display(df[["distancia_m","hora","dia_semana","tipo_via","tiempo_min"]].describe(include="all"))

## 2. Verificación de integridad y fuga de información
El tiempo es una variable derivada. No entrenamos utilizando la velocidad contemporánea como predictor. Las transformaciones de escala y codificación se entrenan **solo** dentro del pipeline de entrenamiento.

In [ ]:
import sys
sys.path.insert(0,str(RAIZ/"backend/scripts/ml"))
from entrenar_regresion_red_osm import validar_y_preparar, dividir_por_aristas
data=validar_y_preparar(df)
partes=dividir_por_aristas(data)
for nombre, parte in partes.items():
    print(nombre,"filas:",len(parte),"aristas:",parte.indice_arista_osm.nunique())

## 3. Entrenamiento reproducible
Ejecuta el script desde la raíz del repositorio antes de cargar métricas. Este notebook **no reentrena** automáticamente para evitar sobrescribir experimentos.

```powershell
python .\backend\scripts\ml\entrenar_regresion_red_osm.py
```

In [ ]:
salida=RAIZ/"backend/datos/ml/modelos_regresion_osm"
metricas_path=salida/"metricas_regresion_osm.json"
if metricas_path.exists():
    res=json.loads(metricas_path.read_text(encoding="utf-8"))
    print("Modelo:",res["modelo"])
    print("Validacion:",res["validacion_regresion"])
    display(pd.DataFrame({"Regresión":res["prueba_regresion"],"Referencia":res["prueba_baseline"]}))
else:
    print("Falta entrenar: ejecuta el comando indicado en la celda anterior.")

## 4. Inspección de errores sobre aristas no vistas
Las curvas y los residuos son *respecto a tiempos estimados del dataset*, no respecto a viajes medidos.

In [ ]:
muestra=salida/"muestra_predicciones_prueba.csv"
if muestra.exists():
    p=pd.read_csv(muestra)
    fig,ax=plt.subplots(figsize=(7,5))
    ax.scatter(p.real_estimado_min,p.predicho_min,s=9,alpha=.35)
    lim=max(p.real_estimado_min.max(),p.predicho_min.max())
    ax.plot([0,lim],[0,lim],linestyle="--")
    ax.set(xlabel="Tiempo objetivo del escenario (min)",ylabel="Tiempo predicho (min)",title="Predicción vs objetivo estimado")
    plt.show()
    fig,ax=plt.subplots(figsize=(7,4))
    ax.hist(p.predicho_min-p.real_estimado_min,bins=35)
    ax.set(xlabel="Error (min)",ylabel="Frecuencia",title="Distribución de errores del modelo")
    plt.show()
else:
    print("Falta muestra de predicciones; entrena primero.")

## 5. Interpretación y siguiente paso
- Reportar MAE, RMSE y R² del conjunto de prueba, además de la referencia.
- Si no se supera la referencia, explicar las limitaciones del modelo lineal y estudiar mejoras.
- Para integrar en el backend, utilizar `predecir()` del script en una función de servicio sin alterar todavía las rutas existentes.
- La validación real requerirá las observaciones históricas reservadas de Bitcarrier y una estrategia justificable de correspondencia espacial.
- Evitar afirmar que el modelo predice tiempos reales de Bogotá con el mismo MAE del simulador.